# Variance in R — Solution Notebook

**Goal:** Learn variance as a measure of spread, compute it step-by-step from first principles, compare population vs sample formulas, visualize two classes with nearly identical means but very different variances, and adapt the story for different audiences.

**Data:** Teacher test-grade vectors (and simple 5-point example). CSVs also available under `data/`.

**Flowchart of desired outcome:**

![Variance Flowchart](variance_in_r_flowchart.png)

---
## Learning Objectives
1. Explain why mean/median alone are insufficient when distributions differ in spread.
2. Manually compute population variance: average of squared deviations from the mean.
3. Implement a reusable `variance()` function and compare it with R's `var()` (sample).
4. Overlay means and interpret the teaching-style story told by variance.
5. Explore alternate implementations, extra practice, and a parameterised Monte-Carlo simulation.
6. Write audience-aware interpretations (analyst vs educator / non-specialist).


## Step 0 — Setup & Inspect the Two Grade Distributions


In [ ]:
teacher_one_grades <- c(83.42, 88.04, 82.12, 85.02, 82.52, 87.47, 84.69, 85.18, 86.29, 85.53, 81.29, 82.54, 83.47, 83.91, 86.83, 88.5, 84.95, 83.79, 84.74, 84.03, 87.62, 81.15, 83.45, 80.24, 82.76, 83.98, 84.95, 83.37, 84.89, 87.29)
teacher_two_grades <- c(85.15, 95.64, 84.73, 71.46, 95.99, 81.61, 86.55, 79.81, 77.06, 92.86, 83.67, 73.63, 90.12, 80.64, 78.46, 76.86, 104.4, 88.53, 74.62, 91.27, 76.53, 94.37, 84.74, 81.84, 97.69, 70.77, 84.44, 88.06, 91.62, 65.82)

cat("Teacher 1: n =", length(teacher_one_grades),
    " mean =", round(mean(teacher_one_grades), 2), "\n")
cat("Teacher 2: n =", length(teacher_two_grades),
    " mean =", round(mean(teacher_two_grades), 2), "\n")
print(summary(teacher_one_grades))
print(summary(teacher_two_grades))

# Overlaid histograms
hist(teacher_one_grades, col = rgb(0,0,1,0.25), xlim = c(65,105),
     main = "Teacher Grades One and Two", xlab = "Grade",
     breaks = 15, border = "blue")
hist(teacher_two_grades, col = rgb(1,0,0,0.25), add = TRUE,
     breaks = 15, border = "red")
legend("topright", c("Teacher 1 (low var)", "Teacher 2 (high var)"),
       fill = c(rgb(0,0,1,0.25), rgb(1,0,0,0.25)))
box()


## Step 1 — Why Variance? Same Mean, Different Spread

Both classes have mean ≈ 84.4. Teacher 1's scores are tightly clustered (consistent mastery). Teacher 2's scores are widely dispersed – some students excel while others are left behind. The x-axis units (percentage points) matter because a 10-point spread is educationally meaningful.


In [ ]:
mean_one <- mean(teacher_one_grades)
mean_two <- mean(teacher_two_grades)
print(paste("Teacher 1 mean:", round(mean_one, 3)))
print(paste("Teacher 2 mean:", round(mean_two, 3)))


## Step 2 — Manual Calculation on a Tiny Example


In [ ]:
grades <- c(88, 82, 85, 84, 90)
mu <- mean(grades)
print(paste("Mean of grades:", mu))

difference_one   <- 88 - mu
difference_two   <- 82 - mu
difference_three <- 85 - mu
difference_four  <- 84 - mu
difference_five  <- 90 - mu

print(c(d1=difference_one, d2=difference_two, d3=difference_three,
        d4=difference_four, d5=difference_five))

# Note that the sum of signed differences is (approximately) zero
print(paste("Sum of diffs:", sum(c(difference_one,difference_two,difference_three,difference_four,difference_five))))

sq1 <- difference_one^2
sq2 <- difference_two^2
sq3 <- difference_three^2
sq4 <- difference_four^2
sq5 <- difference_five^2
print(c(sq1,sq2,sq3,sq4,sq5))

sum_sq <- sq1 + sq2 + sq3 + sq4 + sq5
N <- length(grades)
pop_variance <- sum_sq / N
print(paste("Population variance (manual):", pop_variance))


## Step 3 — The Population Variance Formula


In [ ]:
variance_pop <- function(x) {
  mean( (x - mean(x))^2 )
}

print(variance_pop(grades))
print(variance_pop(teacher_one_grades))
print(variance_pop(teacher_two_grades))


## Step 4 — R's Built-in `var()` (Sample Variance)


In [ ]:
sample_var_grades <- var(grades)
print(paste("Sample variance (var()):", sample_var_grades))
print(paste("Check: pop * N/(N-1) =", variance_pop(grades) * length(grades)/(length(grades)-1)))


## Step 5 — Full Comparison & Visualisation


In [ ]:
summary_df <- data.frame(
  Teacher   = c("One", "Two"),
  Mean      = c(mean(teacher_one_grades), mean(teacher_two_grades)),
  Pop_Var   = c(variance_pop(teacher_one_grades), variance_pop(teacher_two_grades)),
  Sample_Var= c(var(teacher_one_grades), var(teacher_two_grades))
)
print(summary_df)

hist(teacher_one_grades, col = rgb(0,0,1,0.25), xlim = c(65,105),
     main = "Means nearly identical – Variance tells the story",
     xlab = "Grade", breaks = 15)
hist(teacher_two_grades, col = rgb(1,0,0,0.25), add = TRUE, breaks = 15)
abline(v = mean(teacher_one_grades), col = "blue", lwd = 2, lty = 2)
abline(v = mean(teacher_two_grades), col = "red",  lwd = 2, lty = 2)
legend("topright",
       c("Teacher 1", "Teacher 2", "Mean T1", "Mean T2"),
       fill = c(rgb(0,0,1,0.25), rgb(1,0,0,0.25), NA, NA),
       border = c("black","black",NA,NA),
       lty = c(NA,NA,2,2), col = c(NA,NA,"blue","red"), lwd = 2)
box()


## Step 6 — Interpretation (Audience Adaptation)

**Analyst / technical supervisor:**  
Population variances are 4.27 (Teacher 1) versus 78.13 (Teacher 2). The sample variances reported by `var()` are 4.41 and 80.83 respectively (division by $N-1$). The roughly 18-fold difference in variance quantifies the dramatically larger dispersion of Teacher 2's outcomes and is robust to the choice of divisor for $N=30$.

**Educator / school administrator (non-specialist):**  
Both teachers' classes average about 84 %. In Teacher 1's class almost every student scores between 80 and 88 – the results are highly consistent. In Teacher 2's class scores range from the mid-60s to over 100; some students are excelling while others appear to be struggling. Variance makes that difference visible in a single number and suggests different instructional approaches or needs for differentiation.


In [ ]:
cat("=== Analyst version ===\n")
cat("Pop var T1 =", round(variance_pop(teacher_one_grades),2),
    " | Pop var T2 =", round(variance_pop(teacher_two_grades),2), "\n")
cat("Sample var (R var()) T1 =", round(var(teacher_one_grades),2),
    " | T2 =", round(var(teacher_two_grades),2), "\n")

cat("\n=== Educator version ===\n")
cat("Both classes average ~84%. Teacher 1 shows tight, consistent results;\n")
cat("Teacher 2 shows a wide spread – some students ace tests, others lag behind.\n")


## Alternate Implementations


In [ ]:
# Alternate A: explicit sum / length
alt_A <- function(x) {
  m <- mean(x)
  sum( (x - m)^2 ) / length(x)
}

# Alternate B: for-loop
alt_B <- function(x) {
  m <- mean(x)
  s <- 0
  for (xi in x) {
    s <- s + (xi - m)^2
  }
  s / length(x)
}

# Alternate C: sd()^2 gives *sample* variance
alt_C <- function(x) sd(x)^2

print(c(
  pop   = variance_pop(grades),
  A     = alt_A(grades),
  B     = alt_B(grades),
  C_samp= alt_C(grades)
))


## More Practice


In [ ]:
# Practice 1 – even wider spread
set.seed(1)
teacher_three_grades <- rnorm(30, mean = 84, sd = 15)
cat("Teacher 3 mean:", round(mean(teacher_three_grades),2),
    " pop var:", round(variance_pop(teacher_three_grades),2), "\n")

# Practice 2 – subset high performers of Teacher 2
high_only <- teacher_two_grades[teacher_two_grades >= 80]
cat("High-only n:", length(high_only),
    " mean:", round(mean(high_only),2),
    " pop var:", round(variance_pop(high_only),2), "\n")

# Practice 3 – transformation properties
cat("Original pop var:", variance_pop(grades), "\n")
cat("After +10      :", variance_pop(grades + 10), "  (unchanged)\n")
cat("After *2       :", variance_pop(grades * 2),
    "  (≈ original * 4 =", variance_pop(grades)*4, ")\n")


## Simulation Section


In [ ]:
set.seed(42)

# ---- Parameters you can tweak ----
true_mean <- 84
true_sd   <- 3
n         <- 30
n_sims    <- 500
# ----------------------------------

emp_vars <- replicate(n_sims, {
  x <- rnorm(n, mean = true_mean, sd = true_sd)
  variance_pop(x)
})

print(summary(emp_vars))
cat("Theoretical population variance =", true_sd^2, "\n")
cat("Mean of empirical variances     =", mean(emp_vars), "\n")

hist(emp_vars, breaks = 30, col = "steelblue", border = "white",
     main = paste0("Monte-Carlo: empirical pop. variance (true sd=", true_sd, ")"),
     xlab = "Estimated population variance")
abline(v = true_sd^2, col = "red", lwd = 2, lty = 2)
abline(v = mean(emp_vars), col = "darkgreen", lwd = 2)
legend("topright",
       c("Theoretical σ²", "Mean of estimates"),
       col = c("red", "darkgreen"), lty = c(2,1), lwd = 2)


## Key Takeaways

- Variance quantifies **spread / dispersion** around the mean.
- Squaring the deviations makes them **positive** and weights large deviations more heavily.
- Population formula divides by **N**; sample formula (R's `var()`) divides by **N-1**.
- Two data sets can share the same mean yet tell completely different stories once variance is examined.
- Always pair a variance (or SD) number with a visualisation when communicating to non-specialists.
- Adding a constant leaves variance unchanged; multiplying by $k$ multiplies variance by $k^2$.
